# **Pibic-Pibit-Huac**: Classificação de dataset de laudos de Radiografia de Tórax utilizando **Qwen2.5-14B-Instruct**

Este notebook faz parte de um projeto de pesquisa PIBIC/PIBITI, desenvolvido na Universidade Federal de Campina Grande (UFCG) com foco em conceber e avaliar uma ferramenta, baseada em Inteligência Artificial, para classificação e geração de laudos de Radiografia de Tórax produzidos pelo setor de Diagnóstico por Imagem do Hospital Universitário Alcides Carneiro.

## Ferramentas utilizadas
 - [Ollama](https://ollama.com/): ferramenta de código aberto utilizada para executar LLMs em ambiente local, mantendo controle total sobre utilização de hardware e configurações de utilização dos modelos. Neste notebook, utilizamos _Ollama_ majoritariamente para realização de inferências.
 - [Pandas](https://pandas.pydata.org/): ferramenta de código aberto utilizada para manipulação e análise de dados. Utilizada neste notebook para carregamento dos dados do _dataset_ a serem classificados.

## Dataset

O _dataset_ utilizado é mais um fruto do trabalho de pesquisa desenvolvido e responsável pela criação deste notebook. Até a data de desenvolvimento deste notebook, ele está disponível publicamente (mas com download via requisição) no HuggingFace. Você pode consultá-lo a partir deste [link](https://huggingface.co/datasets/guilhermenf/huac_chest_xray_reports).

Esta base de dados contém 101 laudos (apenas texto) de Radiografia de Tórax produzidos pelo HUAC.

## Bônus

Também como parte do trabalho de pesquisa, um dataset alternativo (mas que não será usado neste notebook) foi desenvolvido, dessa vez contendo **laudos e imagens** de Radiografia de Tórax. Você pode consultar - e pedir acesso - a partir deste [link](https://huggingface.co/datasets/guilhermenf/huac_chest_xray_reports_images).

## Autores

**Guilherme Noronha**, graduando em Ciência da Computação pela Universidade Federal de Campina Grande (UFCG).

 - **Email**: guilherme.noronha.fragoso@ccc.ufcg.edu.br
 - **Github**: [guinoronhaf](https://github.com/guinoronhaf)
 - **HuggingFace**: [guilhermenf](https://huggingface.co/guilhermenf)
 - **Linkedin**: [Guilherme Fragoso](https://www.linkedin.com/in/guilherme-noronha-fragoso/)

**João Ventura**, graduando em Ciência da Computação pela Universidade Federal de Campina Grande (UFCG).

 - **Email**: joao.ventura.crispim.neto@ccc.ufcg.edu.br
 - **Github**: [joaoneto9](https://github.com/joaoneto9)
 - **HuggingFace**: [joaneto9](https://huggingface.co/joaoneto9)
 - **Linkedin**: [João Neto](https://www.linkedin.com/in/jo%C3%A3oneto09/)

## Instalando dependências

In [ ]:
# Atualiza dependências do kernel
!sudo apt-get install

# Instala zstd, uma biblioteca de compressão lossless de dados utilizado pelo ollama
!sudo apt install zstd

# Instala libs responsáveis por mapear as configurações de hardware para o ollama
!sudo apt-get install pciutils lshw

# Baixa e executa o binário do ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Necessário para utilização da função chat, mais à frente
!pip install ollama

## Colocando serviço do Ollama no ar

In [ ]:
import subprocess
import time

# Executa serviço do ollama em segundo plano
process = subprocess.Popen(['ollama', 'serve'],
                          stdout=subprocess.PIPE,
                          stderr=subprocess.PIPE)

# Espera para garantir que o serviço subiu
time.sleep(5)

print("ollama server no ar!")

In [ ]:
# Escuta porta padrão na qual o serviço do Ollama roda
!sudo lsof -i :11434

## Baixando modelo

Para baixar um modelo com Ollama, basta executar:

```bash
ollama pull <id_modelo>
```

Isso baixa o modelo no ambiente virtual mantido pelo serviço do ollama.

In [ ]:
# Baixa Qwen2.5-14B-Instruct
!ollama pull qwen2.5:14b

In [ ]:
# Lista os modelos baixados até agora
!ollama list

## Construindo prompt

Vamos, aqui, tirar proveito do _chat_template_ do modelo em questão. Para isso, o prompt será divido em:

 - *system_prompt*: aqui definiremos o tom e os objetivos que deverão ser adotados pelo modelo durante suas respostas.
 - *user_prompt*: aqui transmitiremos o comando direto para o modelo, isto é, o que deve ser feito naquele instante.

Antes disso, no entanto, precisamos definir os critérios de nossa classificação. Isso porque, durante a classificação dos laudos de Radiografia de Tórax, levaremos em consideração 6 critérios ao todo. Esses critérios indicam tópicos que devem ser analisados durante a elaboração de laudos de exames desse tipo. São eles:

1) **Ossos e partes moles**.
2) **Avaliação da Traqueia**.
3) **Pulmões e Pleura**.
4) **Silhueta Cardíaca e Mediastino**.
5) **Dispositivos Externos**.
6) **Alterações anatômicas**.

Para este processo de classificação, consideramos apenas os critérios **1, 3, 4 e 6**. Isso porque os demais critérios são plenamente opcionais do ponto de vista de necessidade de descrição em um laudo de Radiografia de Tórax (produzido pelo HUAC). Dessa forma, concentraremos os esforços do modelo nos critérios fundamentais.

## Metodologia de classificação

A intenção é pedir ao modelo para atribuir notas de 0.0 a 1.0 para cada um dos 4 critérios, junto a uma justificativa que explique o porquê da atribuição de tais notas. Tal classificação será utilizada para **ajuste fino supervisionado** de modelos menores.

A metodologia posterior (que não se enquadra no propósito deste notebook) é somar as notas dos 4 critérios, chegando a um **subtotal**. Este subtotal será somado a 2 (em uma suposição de que os critérios considerados "não obrigatórios" são sempre atendidos quando não estão presentes explicitamente), definindo um **total**. Por fim, este total determinará uma classificação com a seguinte lógica:

 - `total <= 2.5` -> laudo padrão **BRONZE**.
 - `2.5 < total <= 5.5` -> laudo padrão **PRATA**.
 - `total > 5.5` -> laudo padrão **OURO**.

A princípio, esta lógica não será incorporada a este notebook, tendo em vista que o propósito aqui é tão somente a classificação de todo o dataset.

### Critérios (criteria)

In [ ]:
criteria = """
### CRITÉRIO 1 - OSSSO E PARTES MOLES

OBJETIVO:
Verificar se houve avaliação estrutural adequada das estruturas ósseas, cúpulas diafragmáticas e partes moles.

IMPORTANTE:
Caso não sejam verificadas alterações nas estruturas avaliadas, deve-se explicitar a normalidade da observação.

EXEMPLOS QUE PODEM RECEBER 1.0:
- "Arcabouço ósseo visível de aspecto anatômico."
- "Estruturas ósseas visualizadas sem alterações."
- "Costelas e clavículas sem alterações."
- "Fratura em arco costal posterior direito."

NOTA 0.5:
- Avaliação pouco específica do arcabouço ósseo em aspecto anatômico.
OU
- Há menção parcial das estruturas ósseas.

EXEMPLOS QUE PODEM RECEBER 0.5:
"- Osteófitos marginais em corpos vertebrais de coluna dorsal." -> Avaliação pouco específica do arcabouço ósseo.

"- Discretas alterações degenerativas em coluna dorsal." -> Avaliação pouco específica do arcabouço ósseo.

"- Desvio do eixo da coluna torácica com convexidade para a direita.
 - Alterações degenerativas de coluna torácica." -> Avaliação pouco específica do arcabouço ósseo.

"- Alterações degenerativas da coluna dorsal." -> Avaliação pouco específica do arcabouço ósseo.
 

NOTA 0.0:
- Nenhuma menção às estruturas ósseas, partes moles ou cúpulas diafragmáticas.

### CRITÉRIO 2 - PULMÕES E PLEURA

OBJETIVO:
Avaliar a descrição pulmonar e pleural.

ITENS ESPERADOS:
- Opacidades (se houver);
- Consolidações (se houver);
- Nódulos (se houver);
- Massas (se houver);
- Seios costofrênicos.

IMPORTANTE:
- A menção a apenas um dos seios costofrênicos implica normalidade do outro -> pode receber 1.0.

NOTA 1.0:
- Avaliação do pulmão
E
- Avaliação pleural
E
- Avaliação dos seios costofrênicos.

A AVALIAÇÃO PLEURAL PODE SER:
- ausência de derrame pleural;
- presença de derrame;
- espessamento pleural;
- pneumotórax;
- redução da profundidade;
- descrição equivalente.

EXEMPLO NOTA 1.0:
"- Transparência pulmonar preservada.
 - Seios costofrênicos livres."

"- Nódulo hiperdenso em lobo superior direito, medindo 6mm (granuloma calcificado).
 - Seios costofrênicos livres."

"- Obliteração do seio costofrênico esquerdo.
 - Transparência pulmonar preservada." -> Mesmo avaliando apenas um dos seios costofrênicos, deve receber nota 1.0.

NOTA 0.5:
- Descrição de alguma(s) das estruturas, mas não de todas.

NOTA 0.0:
- Sem descrição de nenhuma estrutura.

### CRITÉRIO 3 - SLIHUETA CARDÍACA E MEDIASTINO

OBJETIVO:
Avaliar a descrição cardíaca e mediastinal.

IMPORTANTE:
- Descrições indiretas de alterações mediastinais também contam como avaliação do mediastino.

NOTA 1.0:
- Avaliação formal do Índice Cardiotorácico.
- Avaliação formal do Mediastino.
- Avaliação da Área Cardíaca.
- Avaliação dos hilos pulmonares (se houver alteração).
- Avaliação do botão aórtico (se houver alterações).

EXEMPLOS NOTA 1.0:
"- Mediastino sem alterações.
 - Índice cardiotorácico dentro da normalidade."

"- Placas parietais calcificadas na crossa da aorta.
 - Mediastino sem alterações.
 - Índice cardiotorácico no limite superior da normalidade."

"- Índice cardiotorácico no limite superior da normalidade.
 - Verifica-se, ainda, discreta retração mediastinal." 

NOTA 0.5:
- Avaliação de alguns elementos OBRIGATÓRIOS, mas não de todos.
- Descrição POUCO FORMAL das estruturas.

EXEMPLOS NOTA 0.5:
"- Aorta torácica alongada.
 - Aumento do índice cardiotorácico" -> Não avaliou o MEDIASTINO.

"- Mediastino sem alterações.
 - Imagem cardíaca de configuração e dimensões normais." -> Descrição pouco formal do ICT.

NOTA 0.0:
- Sem menção a nenhuma das estruturas.

IMPORTANTE:
- Em caso de aumento do ICT, não é preciso detalhar a magnitude do aumento.

### CRITÉRIO 4: DESCRIÇÃO DAS ALTERAÇÕES

OBJETIVO:
Avaliar se as alterações descritas possuem caracterização adequada, CASO hajam alterações.

IMPORTANTE:
Este critério deve avaliar apenas alterações QUE EXISTAM no texto do laudo.

- Se não houver alterações descritas, atribua 1.0.
- Se houver alterações descritas:
    - descrição objetiva e localizada -> 1.0.
    - descrição vaga ou sem localização -> 0.5.

ITENS ESPERADOS:
- Descrição objetiva.
- Localização anatômica.

EXEMPLO NOTA 1.0:
- "Osteófitos marginais em corpos vertebrais de coluna dorsal.".
- "Espondiloartrose dorsal."
- "Alterações degenerativas de coluna torácica.".
- "Desvio do eixo da coluna torácica com convexidade para a direita."
"""

print("Critérios de classificação definidos!")

### system_prompt

Este modelo de _system_prompt_ acompanha exemplos de laudos classificados, em uma técnica conhecida como **few-shot**.

In [ ]:
system_prompt = f"""
Você é um Radiologista Analista Sênior especializado em auditoria estrutural de laudos radiológicos.

OBJETIVO:
Avaliar a QUALIDADE ESTRUTURAL de um laudo de radiografia de tórax.

IMPORTANTE:
Você NÃO deve avaliar diagnóstico médico.
Você deve avaliar apenas:
- cobertura estrutural do laudo;
- formalismo técnico;
- especificidade anatômica;
- completude da descrição.

CRITÉRIOS:
{criteria}

REGRAS GERAIS (OBRIGATÓRIAS):

1. NÃO ASSUMA INFORMAÇÕES
- Avalie APENAS o que estiver explicitamente escrito.
- NÃO infira estruturas não mencionadas.
- NÃO considere frases genéricas como equivalentes a avaliação completa.

2. REGRA DE AGREGAÇÃO
Informações distribuídas entre:
- descrição,
- achados,
- conclusão,
- impressão diagnóstica

devem ser combinadas para calcular a nota final.

4. DESCRIÇÕES IMPLÍCITAS ACEITAS
Descrições globais podem ser aceitas quando indicarem explicitamente avaliação estrutural.

EXEMPLOS ACEITOS:
- "Estruturas ósseas visualizadas sem alterações."
- "Arcabouço ósseo sem alterações evidentes."

5. PROCESSO INTERNO (NÃO EXIBIR)
Para cada critério:
- localizar evidências textuais;
- verificar completude;
- verificar formalismo técnico;
- verificar especificidade anatômica;
- atribuir nota.

6. FORMATO DE SAÍDA
Responder APENAS com JSON válido.

NÃO incluir:
- markdown;
- comentários;
- texto fora do JSON.

FORMATO OBRIGATÓRIO:

{{
    c1: pontuacação critério 1,
    c2: pontuacação critério 2,
    c3: pontuacação critério 3,
    c4: pontuacação critério 4,
    justificativa: Justificativa técnica para as pontuações.
}}

EXEMPLOS INTERNOS:

EXEMPLO 1:
Laudo:
- Mínimo derrame pleural à esquerda.
- Transparência pulmonar preservada.
- Mediastino sem alterações.
- Índice cardiotorácico dentro da normalidade.
- Estruturas ósseas visualizadas sem alterações

Saída:
{{
    c1: 1.0,
    c2: 0.5,
    c3: 1.0,
    c4: 1.0,
    justificativa: Não foram identificadas alterações nas estruturas ósseas, o que foi explicitamente informado (critério 1). Não há menção à avaliação dos seios costofrênicos, o que resulta em 0.5 para o critério 2; São avaliados ICT e mediastino, garantindo nota 1.0 para o critério 3; Como não há menção a alterações observadas, infere-se que não existem, o que resulta em nota 1 para o critério 4.
}}


EXEMPLO 2:
Laudo:
- Questiona-se acentuação da trama vascular pulmonar.
- Seios costofrênicos livres.
- Mediastino sem alterações.
- Índice cardiotorácico no limite superior da normalidade.
- Alterações degenerativas da coluna torácica.

Saída:
{{
    c1: 0.5,
    c2: 1.0,
    c3: 1.0,
    c4: 1.0,
    justificativa: Apesar de haver menção a alterações na coluna torácica, a análise das estruturas ósseas não está detalhada; Análise pulmonar e dos seios costofrênicos é adequada; ICT e mediastino foram avaliados corretamente; Há descrição precisa das alterações encontradas.    
}}

EXEMPLO 3:
Laudo:
- Nódulo hiperdenso em lobo superior direito, medindo 6mm (granuloma calcificado).
- Seios costofrênicos livres.
- Índice cardiotorácico dentro da normalidade.
- Discretas alterações degenerativas em coluna dorsal.
- Pequena formação alongada e hiperdensa na projeção do hilo pulmonar direito (clipe metálico?). Verifica-se,
ainda, discreta retração mediastinal. Correlacionar com antecedentes cirúrgicos.

Saída:
{{
    c1: 0.5,
    c2: 1.0,
    c3: 1.0,
    c4: 1.0,
    justificativa: Apesar de haver menção a alterações na coluna dorsal, a avaliação do arcabouço ósseo em aspecto anatômico é pouco específica (c1 = 0.5); A descrição de pulmões, pleura e seios costofrênicos está adequada; Há presença da análise do ICT e do mediastino ("Verifica-se, ainda, discreta retração mediastinal"); Por fim, as alterações encontradas são descritas corretamente, de maneira objetiva e detalhada.
}}

EXEMPLO 4:
Laudo:
- Transparência pulmonar preservada.
- Obliteração do seio costofrênico esquerdo (pequeno derrame pleural?).
- Índice cardiotorácico aumentado.
- Mediastino sem alterações.
- Alterações degenerativas de coluna torácica.
- Assimetria das sombras mamárias. Correlacionar com antecedentes cirúrgicos.

Saída:
{{
    "c1": 0.5,
    "c2": 1.0,
    "c3": 1.0,
    "c4": 1.0,
    "justificativa": Avaliação pouco formal do arcabouço ósseo em aspecto anatômico, o que confere 0.5 ao critério 1; Pulmões e pleural avaliadas adequadamante. A avaliação do seio costofrênico esquerdo apenas implica avaliação e normalidade do seio costofrênico direito, o que resulta em 1.0 para o critério 2; ICT e mediastino são avaliados adequadamente, resultando em 1.0 para o critério 3. Por fim, as alterações existentes são descritas de maneira objetiva e com localização anatômica, conferindo nota 1.0 ao critério 4.
}}
"""

print("System_prompt definido!")

### user_prompt

In [ ]:
"""
Gera o user_prompt a partir de um laudo, em formato de string.

params:
    report (str): laudo a ser fornecido ao modelo para classificação.
returns:
    user_prompt contendo o laudo e as demais instruções de classificação para o modelo.
"""
def generate_user_prompt(report: str) -> str:
    return f"""
AVALIE O SEGUINTE LAUDO RADIOGRÁFICO.

TEXTO DO LAUDO:
--------------------
{report}
--------------------

INSTRUÇÕES:

1. Avalie estritamente conforme os critérios definidos.

2. NÃO assuma estruturas não mencionadas.

3. Penalize:
- baixa especificidade anatômica;
- linguagem pouco técnica;
- ausência de estruturas obrigatórias.

4. Considere descrições implícitas válidas quando houver indicação clara de avaliação estrutural.

5. Utilize SOMENTE as notas:
- 1.0
- 0.5
- 0.0

6. Retorne APENAS JSON válido.

7. NÃO escreva texto fora do JSON.
"""


print("Função para geração do user_prompt definida!")

## Constantes importantes

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

URL_DATASET = "hf://datasets/guilhermenf/huac_chest_xray_reports/" # define a url do dataset
HF_TOKEN = user_secrets.get_secret("HF_TOKEN_GUI") # define o token de acesso do huggingface
SPLITS = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'} # define os splits de acesso aos dados do dataset
MODEL_ID = "qwen2.5:14b" # define o id do modelo no ollama
N_REPORTS = 5 # define o número de laudos a serem analisados de uma só vez
N_LOOPS = 2 # define o número de classificações a serem feitas para cada laudo

print("Constantes definidas!")

## HF Login

In [ ]:
from huggingface_hub import login

# Tenta realizar login na plataforma HuggingFace para download do dataset de laudos
try:
    login(HF_TOKEN)
    print("Login realizado!")
except Except as exc: # Exceção em caso de falha
    print(exc)

## Baixando dataset

In [ ]:
import pandas as pd

df_train = pd.read_parquet(URL_DATASET + SPLITS["train"]) # DataFrame de treino
df_test = pd.read_parquet(URL_DATASET + SPLITS["test"]) # DataFrame de teste

print("Dataframes de treino e teste obtidos!")

In [ ]:
# concatena os datasets em um único DataFrame
df = pd.concat([df_train, df_test], ignore_index=True)

# lista de laudos com list comprehensiopn
reports = [rep for rep in df["report"]][:]
# lista de requisições (identificadores) com list comprehension
requisitions = [req for req in df["requisition"]][:]
# lista os ids seriais artificiais de auxílio de identificação com list comprehension
ids = [artif_id for artif_id in df["id"]][:]

print("Laudos, requisições e IDs extraídos!")

In [ ]:
data = [] # inicializa lista vazia

for report, requisition, serial_id in zip(reports, requisitions, ids): # associa laudos, requisições e IDs
    data.append({ # adiciona objeto à lista data
        "id": serial_id,
        "requisition": requisition,
        "report": report
    })

print("Dicionário de dados definido!")

## Função para realização de inferência

In [ ]:
from ollama import chat

"""
Realiza a inferência de um modelo, retorando uma tupla contendo thinking (raciocínio) do modelo e o conteúdo (content) de sua resposta.

params:
    model_id (str): id do modelo em que a inferência será realizada.
    report (str): laudo a ser avaliado.
returns:
    tupla contendo thinking e content.
"""
def send_prompt_for_inference(model_id: str, report: str):
    response = chat(
        model=model_id,
        messages=[
            {
                'role': 'system',
                'content': system_prompt # system_prompt que definimos
            },
            {
                'role': 'user',
                'content': generate_user_prompt(report) # user_prompt gerado pela função
            }
        ],
    )

    return (vars(response["message"])["thinking"], vars(response["message"])["content"])


print("Função para inferência definida!")

## Realizando inferência

### Funções de apoio à filtragem de informações

Antes de realizar de fato o processo de inferência, é importante definir duas funções que auxiliarão no processo de filtragem de informações após a inferência. São elas:

 - `def extract_text(text: str, pattern: str) -> str`: extrai o conteúdo de `text` a partir de um padrão (regex) `pattern`.
 - `def get_criteria_sum(report_classification: str) -> float`: realiza o somatório das notas atribuídas aos critérios durante a classificação.
 - `def get_label(points: int) -> str`: retorna a _label_ equivalente a partir da quantidade de pontos.

In [ ]:
import re

"""
Extrai conteúdo de um texto a partir de um padrão definido por expressões regulares (regex).

params:
    text (str): texto a ser analisado.
    pattern (str): padrão (regex) no qual a extração de conteúdo é baseada.

returns:
    conteúdo extraído do texto original a partir do padrão.
"""
def extract_text(text: str, pattern: str) -> str:
    response = re.search(pattern, text, re.DOTALL | re.IGNORECASE) # procura pelo padrão pattern no texto
    if response:
        final_response = response.group(1).strip() # extrai group(1)
        return final_response # retorna a extração final
    return None


print("Função para extração de texto com regex definida!")

In [ ]:
import json

"""
Soma as notas atribuídas pelo modelo a cada um dos critérios. Converte-se a string recebida
em um dicionário python e realiza-se o somatório.

params:
    report_classfiication (str): classificação do laudo.
returns:
    somatório das notas atribuídas aos critérios analisados.
"""
def get_criteria_sum(report_classification: str) -> float:
    json_report_classif = json.loads(report_classification) # transforma a string em um json na forma de dicionário python

    sum_criteria = sum(float(value) for key, value in json_report_classif.items() if key.startswith('c') and key[1:].isdijgit()) # soma os valores das chaves cujo texto tem o padrão "cn", onde n é um dígito

    return sum_criteria


print("Função para somatórios dos pontos dos critérios definida!")

In [ ]:
"""
Determina categoria do laudo (OURO, PRATA, BRONZE) a partir de sua pontuação.

params:
    points (float): pontuação do laudo na soma dos critérios.
returns:
    categoria do laudo.
"""
def get_label(points: float) -> str:
    label = "OURO"
    if points <= 2.5:
        label = "BRONZE"
    elif points <= 5.5:
        label = "PRATA"
    return label # retorna a label final


print("Função para determinação da categoria final definida!")

### Processo de inferência

In [ ]:
"""
Recebe uma lista de dicionários, contendo informações específicas de cada laudo, como ID, Requisição do exame e conteúdo do laudo. Em seguida, aplica inferência do modelo
para obter classificação e enquadrar o laudo em uma categoria (OURO, PRATA, BRONZE). Não retorna nada. Apenas modifica o dicionário recebido, adicionando as chaves "label"
e "feedback".

params:
    model_id (str): id do modelo a ser usado para inferência (ollama).
    data (list(dict())): lista de dicionários com os dados de cada laudo.
    n_loops (int): número de inferências a serem feitas para cada laudo.
"""
def classify_reports_dataset(model_id: str, data: list(dict()), n_loops: int = 1):
    feedback_pattern = r'"justificativa"\s*:\s*"(.*?)"' # define a regex de filtragem dos textos, extraindo somente o feedback da classificação do modelo
    points_boost = 2 # boost de 2 pontos citado anteriormente, que assume a pontuação 1.0 para os critérios "desconsiderados" durante esta classificação
    
    for report_data in data:# itera por cada dicionário, extraindo classificação e feedback, e atualizando o dicionário
        report = report_data["report"]
        model_response = send_prompt_for_inference(model_id, report)[1] # obtém apenas o segundo elemento da tupla (que é o conteúdo da resposta do modelo)

        report_label = get_label(get_criteria_sum(model_response) + points_boost) # determina label (categoria) do laudo
        final_model_response = extract_text(model_response, feedback_pattern) # extrai justificativa da classificação

        report_data["label"] = report_label # cria chave "label"
        report_data["feedback"] = final_model_response # cria chave "feecback"

        print("Laudo classificado!") # print para debugging
        print("===============")
        

print("Função para classificação definida!")

In [ ]:
classify_reports_dataset(MODEL_ID, data) # inferência no conjunto de dados (data)

## Armazenando resultados

Armazenaremos o resultado em um arquivo _.json_.

In [ ]:
import json

"""
Escreve lista de dicionários python em arquivo .json.

params:
    data (list(dict())): lista de dicionários python.
    json_path: path do arquivo .json.
"""
def save_data_into_json(data: list(dict()), json_path: str):
    try:
        with open(json_path, 'w') as file:
            json.dump(data, file, indent=4, ensure_ascii=False) # injeta dicionário python na forma de json
        file.close()
        print("Arquivo json gerado!")
    except Exception as e:
        print(e)


print("Função para escrever em arquivo json definfida!")

In [ ]:
save_data_into_json(data, "/kaggle/working/teste.json") # salva o conjunto de dados em um json